# Exploratory Data Analysis (EDA) — Mi Spotify Wrapped

**Estudiantes:** William & Assistant
**Materia:** Bases de Datos II

Este notebook se conecta al Data Warehouse en Neon para analizar los hábitos de escucha extraídos mediante el pipeline ETL.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv('../.env')
db_url = os.getenv('DATABASE_URL')
engine = create_engine(db_url)

print("Conexión exitosa al DWH")

## 1. Carga y Revisión de Datos

In [ ]:
df_history = pd.read_sql('SELECT * FROM dwh.fact_listening_history', engine)
df_artists = pd.read_sql('SELECT * FROM dwh.dim_artists', engine)
df_tracks = pd.read_sql('SELECT * FROM dwh.dim_tracks', engine)

print(f"Registros en historial: {len(df_history)}")
df_history.info()

## 2. ¿Cuándo escucho música?

### Heatmap por Hora y Día de la Semana

In [ ]:
df_history['played_at'] = pd.to_datetime(df_history['played_at'])
df_history['hour'] = df_history['played_at'].dt.hour
df_history['day'] = df_history['played_at'].dt.day_name()

heatmap_data = df_history.groupby(['day', 'hour']).size().unstack(fill_value=0)
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = heatmap_data.reindex(days_order)

plt.figure(figsize=(15, 8))
sns.heatmap(heatmap_data, cmap='Greens', annot=True, fmt='d')
plt.title('Intensidad de Escucha por Día y Hora')
plt.show()

## 3. Artistas y Géneros Dominantes

### Análisis de Pareto

In [ ]:
artist_counts = df_history.merge(df_artists, on='artist_id')['name'].value_counts()
pareto_df = artist_counts.to_frame(name='count')
pareto_df['cumulative_perc'] = pareto_df['count'].cumsum() / pareto_df['count'].sum() * 100

plt.figure(figsize=(12, 6))
sns.lineplot(x=range(len(pareto_df)), y=pareto_df['cumulative_perc'])
plt.axhline(80, color='red', linestyle='--')
plt.title('Curva de Pareto de Artistas')
plt.ylabel('% Acumulado de Reproducciones')
plt.show()

## 4. Popularidad de mi Música

Clasificación: Underground (<30), Emerging (30-60), Mainstream (60-80), Viral (>80).

In [ ]:
def classify_popularity(p):
    if p < 30: return 'Underground'
    if p < 60: return 'Emerging'
    if p < 80: return 'Mainstream'
    return 'Viral'

df_tracks['category'] = df_tracks['popularity'].apply(classify_popularity)
sns.countplot(data=df_tracks, x='category', order=['Underground', 'Emerging', 'Mainstream', 'Viral'])
plt.title('Distribución de Popularidad de mis Canciones')
plt.show()